# File-to-File Comparison Utility

This notebook compares two large files even when the rows are not in the same order.

It supports:
- Excel and CSV files
- Multiple Excel sheets
- Composite-key matching
- Duplicate-key handling using occurrence numbers
- Column-level mismatch reporting
- Records present only in File 1 or File 2
- Excel summary report generation


## 1. Configuration

Update the file paths and matching columns below.

`KEY_COLUMNS` should contain Order Number and the additional columns used to identify the same business record in both files.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Input files: .xlsx, .xls, or .csv
FILE_1 = Path('file_1.xlsx')
FILE_2 = Path('file_2.xlsx')

# Output comparison report
OUTPUT_FILE = Path('file_comparison_report.xlsx')

# Example: ['ORDER_NUMBER', 'ACCOUNT_NUMBER', 'TRADE_DATE']
KEY_COLUMNS = ['ORDER_NUMBER']

# Optional columns to ignore during value comparison
IGNORE_COLUMNS = []

# Normalization options
TRIM_TEXT = True
CASE_INSENSITIVE = False
ROUND_DECIMALS = 8

# For CSV files
CSV_DELIMITER = ','
CSV_ENCODING = 'utf-8'


## 2. Utility Functions

In [ ]:
def load_file(path: Path):
    """Return a dictionary of sheet_name -> DataFrame."""
    suffix = path.suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path, sheet_name=None, dtype=object)
    if suffix == '.csv':
        return {
            'CSV': pd.read_csv(
                path,
                sep=CSV_DELIMITER,
                encoding=CSV_ENCODING,
                dtype=object,
                keep_default_na=False
            )
        }
    raise ValueError(f'Unsupported file type: {suffix}')


def normalize_scalar(value):
    """Normalize a single value for reliable comparison."""
    if pd.isna(value):
        return ''

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, (int, np.integer)):
        return str(value)

    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return ''
        return str(round(float(value), ROUND_DECIMALS))

    text = str(value)
    if TRIM_TEXT:
        text = text.strip()
    if CASE_INSENSITIVE:
        text = text.casefold()
    return text


def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    normalized = df.copy()
    normalized.columns = [str(col).strip() for col in normalized.columns]
    for column in normalized.columns:
        normalized[column] = normalized[column].map(normalize_scalar)
    return normalized


def validate_columns(df1, df2, key_columns):
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)

    missing_in_file_2 = sorted(cols1 - cols2)
    missing_in_file_1 = sorted(cols2 - cols1)
    missing_keys_1 = sorted(set(key_columns) - cols1)
    missing_keys_2 = sorted(set(key_columns) - cols2)

    if missing_keys_1 or missing_keys_2:
        raise ValueError(
            f'Missing key columns. File 1: {missing_keys_1}; File 2: {missing_keys_2}'
        )

    return missing_in_file_1, missing_in_file_2


def add_duplicate_sequence(df: pd.DataFrame, key_columns):
    """Create a deterministic occurrence number for duplicate composite keys."""
    result = df.copy()
    non_key_columns = sorted([c for c in result.columns if c not in key_columns])
    sort_columns = key_columns + non_key_columns
    result = result.sort_values(sort_columns, kind='mergesort').reset_index(drop=True)
    result['_DUPLICATE_SEQUENCE'] = result.groupby(key_columns, dropna=False).cumcount() + 1
    return result


## 3. Sheet Comparison Logic

In [ ]:
def compare_sheet(df1, df2, sheet_name, key_columns, ignore_columns=None):
    ignore_columns = ignore_columns or []

    df1 = normalize_dataframe(df1)
    df2 = normalize_dataframe(df2)

    missing_in_file_1, missing_in_file_2 = validate_columns(df1, df2, key_columns)

    common_columns = [c for c in df1.columns if c in df2.columns]
    compare_columns = [
        c for c in common_columns
        if c not in set(key_columns) | set(ignore_columns)
    ]

    df1 = add_duplicate_sequence(df1[common_columns], key_columns)
    df2 = add_duplicate_sequence(df2[common_columns], key_columns)

    join_columns = key_columns + ['_DUPLICATE_SEQUENCE']

    merged = df1.merge(
        df2,
        on=join_columns,
        how='outer',
        suffixes=('_FILE_1', '_FILE_2'),
        indicator=True
    )

    only_file_1 = merged[merged['_merge'] == 'left_only'].copy()
    only_file_2 = merged[merged['_merge'] == 'right_only'].copy()
    matched_candidates = merged[merged['_merge'] == 'both'].copy()

    mismatch_rows = []
    matched_mask = pd.Series(True, index=matched_candidates.index)

    for column in compare_columns:
        col1 = f'{column}_FILE_1'
        col2 = f'{column}_FILE_2'
        unequal = matched_candidates[col1] != matched_candidates[col2]
        matched_mask &= ~unequal

        if unequal.any():
            temp = matched_candidates.loc[unequal, join_columns + [col1, col2]].copy()
            temp.insert(0, 'SHEET_NAME', sheet_name)
            temp['COLUMN_NAME'] = column
            temp = temp.rename(columns={
                col1: 'FILE_1_VALUE',
                col2: 'FILE_2_VALUE'
            })
            mismatch_rows.append(temp)

    mismatches = (
        pd.concat(mismatch_rows, ignore_index=True)
        if mismatch_rows
        else pd.DataFrame(columns=[
            'SHEET_NAME', *join_columns,
            'FILE_1_VALUE', 'FILE_2_VALUE', 'COLUMN_NAME'
        ])
    )

    fully_matched = matched_candidates.loc[matched_mask].copy()
    mismatch_record_count = len(matched_candidates) - len(fully_matched)

    summary = {
        'SHEET_NAME': sheet_name,
        'FILE_1_RECORDS': len(df1),
        'FILE_2_RECORDS': len(df2),
        'FULLY_MATCHED_RECORDS': len(fully_matched),
        'MISMATCHED_RECORDS': mismatch_record_count,
        'ONLY_IN_FILE_1': len(only_file_1),
        'ONLY_IN_FILE_2': len(only_file_2),
        'COLUMN_MISMATCH_COUNT': len(mismatches),
        'MISSING_COLUMNS_IN_FILE_1': ', '.join(missing_in_file_1),
        'MISSING_COLUMNS_IN_FILE_2': ', '.join(missing_in_file_2)
    }

    denominator = max(len(df1), len(df2), 1)
    summary['MATCH_PERCENTAGE'] = round(
        (len(fully_matched) / denominator) * 100, 4
    )

    for frame in (only_file_1, only_file_2, fully_matched):
        if not frame.empty:
            frame.insert(0, 'SHEET_NAME', sheet_name)

    return {
        'summary': summary,
        'mismatches': mismatches,
        'only_file_1': only_file_1,
        'only_file_2': only_file_2,
        'fully_matched': fully_matched
    }


## 4. Run Comparison for All Sheets

In [ ]:
files_1 = load_file(FILE_1)
files_2 = load_file(FILE_2)

all_sheet_names = sorted(set(files_1) | set(files_2))

summaries = []
all_mismatches = []
all_only_file_1 = []
all_only_file_2 = []
all_fully_matched = []

for sheet_name in all_sheet_names:
    print(f'Comparing sheet: {sheet_name}')

    if sheet_name not in files_1:
        frame = normalize_dataframe(files_2[sheet_name])
        frame.insert(0, 'SHEET_NAME', sheet_name)
        all_only_file_2.append(frame)
        summaries.append({
            'SHEET_NAME': sheet_name,
            'FILE_1_RECORDS': 0,
            'FILE_2_RECORDS': len(frame),
            'FULLY_MATCHED_RECORDS': 0,
            'MISMATCHED_RECORDS': 0,
            'ONLY_IN_FILE_1': 0,
            'ONLY_IN_FILE_2': len(frame),
            'COLUMN_MISMATCH_COUNT': 0,
            'MATCH_PERCENTAGE': 0,
            'MISSING_COLUMNS_IN_FILE_1': 'Entire sheet missing',
            'MISSING_COLUMNS_IN_FILE_2': ''
        })
        continue

    if sheet_name not in files_2:
        frame = normalize_dataframe(files_1[sheet_name])
        frame.insert(0, 'SHEET_NAME', sheet_name)
        all_only_file_1.append(frame)
        summaries.append({
            'SHEET_NAME': sheet_name,
            'FILE_1_RECORDS': len(frame),
            'FILE_2_RECORDS': 0,
            'FULLY_MATCHED_RECORDS': 0,
            'MISMATCHED_RECORDS': 0,
            'ONLY_IN_FILE_1': len(frame),
            'ONLY_IN_FILE_2': 0,
            'COLUMN_MISMATCH_COUNT': 0,
            'MATCH_PERCENTAGE': 0,
            'MISSING_COLUMNS_IN_FILE_1': '',
            'MISSING_COLUMNS_IN_FILE_2': 'Entire sheet missing'
        })
        continue

    result = compare_sheet(
        files_1[sheet_name],
        files_2[sheet_name],
        sheet_name,
        KEY_COLUMNS,
        IGNORE_COLUMNS
    )

    summaries.append(result['summary'])
    if not result['mismatches'].empty:
        all_mismatches.append(result['mismatches'])
    if not result['only_file_1'].empty:
        all_only_file_1.append(result['only_file_1'])
    if not result['only_file_2'].empty:
        all_only_file_2.append(result['only_file_2'])
    if not result['fully_matched'].empty:
        all_fully_matched.append(result['fully_matched'])

summary_df = pd.DataFrame(summaries)
mismatches_df = pd.concat(all_mismatches, ignore_index=True) if all_mismatches else pd.DataFrame()
only_file_1_df = pd.concat(all_only_file_1, ignore_index=True) if all_only_file_1 else pd.DataFrame()
only_file_2_df = pd.concat(all_only_file_2, ignore_index=True) if all_only_file_2 else pd.DataFrame()
fully_matched_df = pd.concat(all_fully_matched, ignore_index=True) if all_fully_matched else pd.DataFrame()

summary_df

## 5. Export Results to Excel

In [ ]:
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

    if not mismatches_df.empty:
        mismatches_df.to_excel(writer, sheet_name='Column_Mismatches', index=False)

    if not only_file_1_df.empty:
        only_file_1_df.to_excel(writer, sheet_name='Only_in_File_1', index=False)

    if not only_file_2_df.empty:
        only_file_2_df.to_excel(writer, sheet_name='Only_in_File_2', index=False)

    if not fully_matched_df.empty:
        fully_matched_df.to_excel(writer, sheet_name='Fully_Matched', index=False)

print(f'Comparison completed. Report created at: {OUTPUT_FILE.resolve()}')

## Output Explanation

- **Summary**: Record counts, matched rows, mismatches, missing rows, and match percentage by sheet.
- **Column_Mismatches**: One output row for every differing column.
- **Only_in_File_1**: Records found only in the first file.
- **Only_in_File_2**: Records found only in the second file.
- **Fully_Matched**: Records where all compared columns match.

For duplicate Order Numbers, add additional business columns to `KEY_COLUMNS`. The notebook also assigns an occurrence number to duplicate composite keys so repeated records can be compared deterministically.